# İzge — RF-DETR-Seg Small insan etiketli eğitim hazırlığı

Hazır, **eğitim çalıştırılmadı**. Bu notebook Mac çıkarım ortamından ayrı Linux x86_64 / Python 3.12.11 / CUDA 12.6 kilidini kullanır. Ham uzun videoları yüklemez. Mevcut pilot etiketsizdir; gerçek etiket/split kabulünden önce eğitim hücreleri kapalı kalır.

## Amaç

İnsan kutu/instance mask etiketleriyle Seg Small uyarlaması. Duruş/feeding sınıflandırıcısı ayrı iştir; bu notebook bunları eğitmez. Model önerileri insan etiketi değildir. [Resmî eğitim API](https://rfdetr.roboflow.com/learn/train/), [Colab koşulları](https://research.google.com/colaboratory/faq.html).

## Kurulum

Colab menüsünden GPU runtime seçin. Önce kullanıcı hesabındaki hedefi belirleyin. Yerel `training-kit.zip` ve **sonradan insan etiketleriyle export edilmiş** COCO zip dosyasını Colab dosya panelinden kendiniz yükleyin. Notebook paylaşılırsa hücre çıktıları da paylaşılabilir; token/anahtar yazmayın. SSH veya uzak worker kullanılmaz.

In [ ]:
from pathlib import Path
KIT_ZIP = Path("/content/training-kit-final.zip")
KIT_SHA256 = ""  # yerel kit manifestindeki SHA256
DATASET_ZIP = Path("/content/human-coco.zip")
DATASET_SHA256 = ""  # kontrollü export paketinin SHA256
KIT_DIR = Path("/content/izge-training-kit")
DATASET_DIR = Path("/content/izge-human-coco")
INSTALL_ENVIRONMENT = False
MOUNT_DRIVE = False
RUN_SMOKE = False
RUN_TRAINING = False
RELEASE_LOCKED_TEST = False
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 4
EPOCHS = 20
OUTPUT_BASE = Path("/content/drive/MyDrive/izge-training")
RESUME_CHECKPOINT = None  # güvenilir tam checkpoint_{epoch}.ckpt; best .pth değil

### 1. Paket checksum ve güvenli açma

Yalnız seçilmiş etiketli görüntüler ve köken manifestleri taşınır. Kod kitinde video, ağırlık ve secret bulunmaz. Dolu hedefler ezilmez.

In [ ]:
import hashlib
import json
import zipfile
def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()
def extract_checked(archive, expected, destination):
    if not expected or digest(archive) != expected:
        raise ValueError("Paket checksum eşleşmiyor")
    if destination.exists():
        raise ValueError("Yeni/boş hedef seçin; önceki veri ezilmez")
    with zipfile.ZipFile(archive) as z:
        entries = z.infolist()
        if len(entries) > 50000 or sum(i.file_size for i in entries) > 20*1024**3:
            raise ValueError("Paket bütçesi aşıldı")
        for i in entries:
            path = (destination/i.filename).resolve()
            if not path.is_relative_to(destination.resolve()) or ((i.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError("Güvensiz paket yolu")
        z.extractall(destination)
extract_checked(KIT_ZIP, KIT_SHA256, KIT_DIR)
extract_checked(DATASET_ZIP, DATASET_SHA256, DATASET_DIR)
proof = json.loads((DATASET_DIR/"dataset_manifest.json").read_text())
if proof.get("human_reviewed") is not True or proof.get("split_verified") is not True or proof.get("task") != "segmentation":
    raise ValueError("İnsan instance mask etiketleri ve bağımsız split kabulü olmadan eğitim kapalı")

### 2. Ayrı kilitli Linux ortamı

Ana Colab Python paketleri üzerine model ortamı kurulmaz; ayrı uv venv kullanılır. Kilit Python 3.12 ve Linux x86_64 içindir. CPU/MPS çıkarım lock’u eğitim lock’u yerine kullanılmaz. Supervision/OpenCV/Roboflow servis istemcisi bu yerel COCO/torchvision eğitim yolunda dışlanmıştır; log/grid/servis özellikleri açılmamalı. Kurulum birkaç GB CUDA wheel indirebilir.

In [ ]:
import subprocess
import sys
import platform
PROJECT = KIT_DIR/"experiments/rfdetr_training"
PYTHON = PROJECT/".venv/bin/python"
def run(args):
    subprocess.run([str(a) for a in args], check=True)
if INSTALL_ENVIRONMENT:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Linux x86_64 gerekli")
    run([sys.executable,"-m","pip","install","uv==0.8.4"])
    run(["uv","sync","--locked","--project",PROJECT,"--python","3.12.11"])
else:
    print("Kurulum çalıştırılmadı; INSTALL_ENVIRONMENT=False")

## Adımlar

### 3. İnsan etiketleri, split ve gerçek loader/GPU kontrolü

COCO cow category_id=1, `dataset_file="roboflow"` yükleyicisinde label=0 olur; önceki COCO-pretrained modelin cow=21 değeriyle karıştırmayın. Aynı oturum/türev/gün farklı split’e geçemez. GPU tipi/belleği yalnız runtime’da öğrenilir; batch=1’in sığacağı henüz ölçülmedi.

In [ ]:
if PYTHON.exists():
    run([PYTHON,KIT_DIR/"scripts/export_behavior_annotations.py","--package",DATASET_DIR,"--validate-export","--task","segmentation"])
    run([PYTHON,PROJECT/"train.py","--check-environment","--report","/content/training-environment.json"])
    gpu_probe = 'import torch, json; print(json.dumps({"cuda": torch.cuda.is_available(), "name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "vram_bytes": torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None}))'
    run([PYTHON,"-c",gpu_probe])
else:
    print("Ayrı ortam henüz kurulmadı")

### 4. Kalıcı checkpoint hedefi ve resmî ağırlık

Drive mount yalnız seçtiğiniz hesap/klasör için yapılır. Eğitim çıktısı doğrudan kalıcı klasöre yazılır; her epoch tam `.ckpt` saklanır. Kesinti sonrası yeniden aynı dataset checksum/kilit ile açıp tam checkpoint’i `resume` olarak seçin. Ağırlık checksum’ı sabittir; indirme eğitim anlamına gelmez.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
WEIGHTS = KIT_DIR/"models/rfdetr/rf-detr-seg-small.pt"
if PYTHON.exists() and (RUN_SMOKE or RUN_TRAINING):
    if not MOUNT_DRIVE or not OUTPUT_BASE.parent.exists():
        raise ValueError("Kalıcı çıktı hedefini seçin")
    run([PYTHON,KIT_DIR/"scripts/fetch_behavior_model.py","--output",WEIGHTS])

### 5. İnsan etiketleri hazırsa bir epoch smoke

Bu hücre bugün çalıştırılmadı. Yükleyici/loss/backward/validation/checkpoint ve CUDA peak bellek ölçülür. OOM olursa durun; micro-batch’i azaltın, accumulation’ı açık tutun. Batch=1 de sığmıyorsa çözünürlük/model veya cihaz kararını ayrıca değerlendirin; otomatik başarı/fallback yok.

In [ ]:
BASE_ARGS = [PYTHON,PROJECT/"train.py","--allow-training","--dataset",DATASET_DIR,"--weights",WEIGHTS,
             "--batch-size",BATCH_SIZE,"--grad-accum-steps",GRAD_ACCUM_STEPS]
if RUN_SMOKE:
    run([*BASE_ARGS,"--smoke","--output-dir",OUTPUT_BASE/"smoke-unique-run"])
else:
    print("Smoke eğitim çalıştırılmadı")

### 6. Eğitim ve resume

Validation model/erken durdurma için kullanılır; test kapalıdır. DINOv3 erişimi veya duruş eğitimi bu model yerine sessizce eklenmez.

In [ ]:
if RUN_TRAINING:
    args = [*BASE_ARGS,"--epochs",EPOCHS,"--output-dir",OUTPUT_BASE/"train-unique-run"]
    if RESUME_CHECKPOINT:
        args += ["--resume",RESUME_CHECKPOINT]
    if RELEASE_LOCKED_TEST:
        args += ["--evaluate-test"]
    run(args)
else:
    print("Gerçek eğitim çalıştırılmadı; RUN_TRAINING=False")

## Kontroller

### 7. Metrikler ve artifact export

`run_report.json`, CSV logları, `training_config.json`, tam `.ckpt` ve seçilen `.pth` dosyalarını dataset/model/ortam manifestleriyle birlikte saklayın. `val` metrikleri saha/test başarısı değildir. Testi yalnız kilitli protokolün son değerlendirmesinde açın; test sonucu eşik ayarlamak için kullanılmaz.

In [ ]:
import shutil
if RUN_SMOKE or RUN_TRAINING:
    for folder in OUTPUT_BASE.iterdir():
        if folder.is_dir():
            files = {str(p.relative_to(folder)):digest(p) for p in folder.rglob("*") if p.is_file()}
            (folder/"artifact_checksums.json").write_text(json.dumps(files,indent=2))
            shutil.make_archive("/content/"+folder.name+"-artifacts","zip",folder)
    print("Zip dosyalarını Colab dosya panelinden indirin; kalıcı checkpoint klasörünü koruyun")

## Sonraki adımlar

Notebook JSON/AST ve yerel Linux import/config/loader kontrolü ayrı kayıtlıdır. **Bu notebook Colab’da top-to-bottom çalıştırılmadı; hesap, GPU ve gerçek etiketli bağımsız split eksik.** Ücretsiz GPU/süre garantili değildir. Eğitim bitince Runtime → Disconnect and delete runtime; checkpoint/paket SHA256 kaydını koruyun. Colab dışı Linux/CUDA tek-GPU seçenekleri `docs/TRAINING_COMPUTE_PLAN.md` içinde.